<a href="https://colab.research.google.com/github/Aris-1712/LLM-fine-tuning/blob/main/Instructional_fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# install dependencies

!pip install transformers
!pip install datasets
!pip install accelerate
!pip install peft
!pip install trl
!pip install bitsandbytes
!pip install pyMuPdf

In [ ]:
# Import exisisting Non Instructional Trained Model

import zipfile

# Open the ZIP file in read mode ('r')
with zipfile.ZipFile('tinyllama-lora_zipped.zip', 'r') as zip_ref:
    # Extract all contents to the current directory
    zip_ref.extractall('tinyllama-lora')

In [ ]:
# load tokenzier to convert strint to tokens

from transformers import AutoTokenizer
model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

# tokenizer should be of the model itself
tokenizer = AutoTokenizer.from_pretrained(model)

# set pad token and eos (end of sentence) token
if tokenizer.pad_token == None:
  tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# Load the Base Model

from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(model, device_map="auto")


In [ ]:
# Load the dataset

from datasets import load_dataset

data = load_dataset("json", data_files="/content/InstructionalTrainingData.json", split= "train")

In [ ]:
def dataPrepare(dataset):
  prompt = f"### Instruction: Answer the question concisely \n### Input:\n{dataset['question']}\n### Response:\n{dataset['answer']}<END>"
  return {"text": prompt}

In [ ]:
# Instructional Training Dataset

finalPrompts = data.map(dataPrepare)
finalPrompts[0]

In [ ]:
# function to tokenize strings and add labels
#  why we need labels: Labels are used to identify the next token while the model is being trained so label has to be there

def tokenize_fn(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    tokens["labels"] = tokens["input_ids"].copy()
    for label in tokens["labels"]:
        if label == tokenizer.pad_token_id:
            label = -100
    return tokens

In [ ]:
finalDataset = finalPrompts.map(tokenize_fn, batched=True)

In [ ]:
from peft import LoraConfig, TaskType
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none"
)

In [ ]:
# crete the peft model using the lora config

from peft import get_peft_model, PeftModel

path = "/content/tinyllama-lora/checkpoint-69"

# Load the base model and non instructional model by merging the weights
nonInstructionTrainingModel = PeftModel.from_pretrained(model, path)
nonInstructionTrainingModel = nonInstructionTrainingModel.merge_and_unload()

peft_model = get_peft_model(nonInstructionTrainingModel, lora_config)

In [ ]:
from transformers import TrainingArguments
args = TrainingArguments(
    output_dir="./tinyllama-lora/instructionalTraining",
    num_train_epochs=4,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none"
)

In [ ]:
# start the training

from transformers import Trainer

trainer = Trainer(
    model=peft_model,
    args=args,
    train_dataset=finalDataset
)

trainer.train()

In [ ]:
from transformers import AutoModelForCausalLM

pathInstructional = "/content/tinyllama-lora/instructionalTraining/checkpoint-16"

modelInstructional = AutoModelForCausalLM.from_pretrained(pathInstructional, device_map="auto")


In [ ]:

prompt = "What was Alphabet's operating income for the first six months of 2025?"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = modelInstructional.generate(
    **inputs,
    max_new_tokens=200, # Increased max_new_tokens
    temperature=0.7,    # Decreased temperature for less randomness
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)


decoded_output = tokenizer.decode(outputs[0],skip_special_tokens=True) # Removed skip_special_tokens=True
print(f"\nDecoded Output (including special tokens):\n{decoded_output}")